In [2]:
import pandas as pd

# Q1

## Códigos
- A10BB01	Glibenclamida
- A10BB09	Gliclazida
- A10BB12	Glimepirida


In [59]:
codigos = ['A10BB01', 'A10BB09', 'A10BB12']
prescricoes = pd.read_csv('prescricoes.csv')

len(prescricoes[prescricoes['codigo_atc'].isin(codigos)])

3035

# Q2

In [3]:
base = pd.read_csv('base_caso_controle.csv')

media_idade_controle = base[base['grupo'] == 'controle']['idade'].mean()
media_idade_caso = base[base['grupo'] == 'caso']['idade'].mean()

print(f"Média do grupo controle: {media_idade_controle:.2f}")
print(f"Média do grupo caso: {media_idade_caso:.2f}")

# Razão DRC Controle

quantidade_controle = len(base[base['grupo'] == 'controle'])
quantidade_controle_com_drc = len(base[base['grupo'] == 'controle'][base['drc'] == 1])

razao_drc_controle = ((quantidade_controle_com_drc / quantidade_controle) * 100)

print(f"Razão de DRC para o grupo de controle: {razao_drc_controle: .2f}%")

# Razão DRC Caso

quantidade_caso = len(base[base['grupo'] == 'caso'])
quantidade_caso_com_drc = len(base[base['grupo'] == 'caso'][base['drc'] == 1])

razao_drc_caso = ((quantidade_caso_com_drc / quantidade_caso) * 100)

print(f"Razão de DRC para o grupo de caso: {razao_drc_caso: .2f}%")

Média do grupo controle: 63.30
Média do grupo caso: 61.53
Razão de DRC para o grupo de controle:  10.63%
Razão de DRC para o grupo de caso:  21.65%


C:\Users\Vitor Bruno\AppData\Local\Temp\ipykernel_12700\2705662672.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  quantidade_controle_com_drc = len(base[base['grupo'] == 'controle'][base['drc'] == 1])
C:\Users\Vitor Bruno\AppData\Local\Temp\ipykernel_12700\2705662672.py:21: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  quantidade_caso_com_drc = len(base[base['grupo'] == 'caso'][base['drc'] == 1])


# Q3

In [61]:
base['data_indice'] = pd.to_datetime(base['data_indice'])
prescricoes['data_prescricao'] = pd.to_datetime(prescricoes['data_prescricao'])

sulfonilureias = prescricoes[prescricoes['codigo_atc'].isin(codigos)].copy()

def definir_exposicao(linha):
    delta_tempo = linha['data_indice'] - pd.Timedelta(days=90)

    receitas_validas = sulfonilureias[
        (sulfonilureias['id_paciente'] == linha['id_paciente']) &
        (sulfonilureias["data_prescricao"] >= delta_tempo) &
        (sulfonilureias["data_prescricao"] < linha['data_indice'])
    ]

    return int(not receitas_validas.empty)

base['exposto'] = base.apply(definir_exposicao, axis=1)

print(base[["id_paciente", "grupo", "data_indice", "exposto"]].head())

prevalencia = base.groupby("grupo")["exposto"].mean() * 100

print("\nPrevalência de exposição:")
print(prevalencia.round(2))



  id_paciente     grupo data_indice  exposto
0      P01297  controle  2022-01-05        0
1      P01088      caso  2022-01-05        0
2      P01894  controle  2022-01-05        0
3      P01601  controle  2022-01-05        0
4      P02019  controle  2022-01-05        0

Prevalência de exposição:
grupo
caso        30.71
controle    12.50
Name: exposto, dtype: float64


# Q4

In [62]:
import math

caso_exposto = len(base[(base['exposto'] == 1) & (base['grupo'] == 'caso')])
caso_nao_exposto = len(base[(base['exposto'] == 0) & (base['grupo'] == 'caso')])

controle_exposto = len(base[(base['exposto'] == 1) & (base['grupo'] == 'controle')])
controle_nao_exposto = len(base[(base['exposto'] == 0) & (base['grupo'] == 'controle')])

print(caso_exposto)
print(caso_nao_exposto)
print(controle_exposto)
print(controle_nao_exposto)

valor_OR_bruta = (caso_exposto * controle_nao_exposto) / (controle_exposto * caso_nao_exposto)

print(f"OR: {valor_OR_bruta: .2f}")

erro_padrao = math.sqrt((1/caso_exposto + 1/caso_nao_exposto + 1/controle_nao_exposto + 1/controle_exposto))

limite_inferior = math.exp(math.log(valor_OR_bruta) - 1.96 * erro_padrao)
limite_superior = math.exp(math.log(valor_OR_bruta) + 1.96 * erro_padrao)

print(f"IC de 95%: {limite_inferior:.2f} - {limite_superior:.2f}")

78
176
127
889
OR:  3.10
IC de 95%: 2.24 - 4.29


# Q5

In [63]:
def definir_exposicao_q5(linha):

    receitas_validas = sulfonilureias[
        sulfonilureias['id_paciente'] == linha['id_paciente']
    ]

    return int(not receitas_validas.empty)

base['exposto_q5'] = base.apply(definir_exposicao_q5, axis=1)

caso_exposto = len(base[(base['exposto_q5'] == 1) & (base['grupo'] == 'caso')])
caso_nao_exposto = len(base[(base['exposto_q5'] == 0) & (base['grupo'] == 'caso')])

controle_exposto = len(base[(base['exposto_q5'] == 1) & (base['grupo'] == 'controle')])
controle_nao_exposto = len(base[(base['exposto_q5'] == 0) & (base['grupo'] == 'controle')])

print(caso_exposto)
print(caso_nao_exposto)
print(controle_exposto)
print(controle_nao_exposto)

valor_OR_q5 = (caso_exposto * controle_nao_exposto) / (controle_exposto * caso_nao_exposto)

print(f"OR: {valor_OR_q5: .2f}")



103
151
299
717
OR:  1.64


# Q6

In [64]:
# Rodar Q4 de novo antes de rodar Q6 caso tenha rodado Q5

drc_por_grupo = base.groupby("grupo")["drc"].mean() * 100

print(drc_por_grupo.round(2))

exposicao_por_drc = base.groupby('drc')['exposto'].mean() * 100

print(exposicao_por_drc.round(2))

sem_drc = base[base["drc"] == 0]
com_drc = base[base["drc"] == 1]

def calcular_or(dados):
    a = len(dados[
        (dados["grupo"] == "caso") &
        (dados["exposto"] == 1)
    ])

    b = len(dados[
        (dados["grupo"] == "controle") &
        (dados["exposto"] == 1)
    ])

    c = len(dados[
        (dados["grupo"] == "caso") &
        (dados["exposto"] == 0)
    ])

    d = len(dados[
        (dados["grupo"] == "controle") &
        (dados["exposto"] == 0)
    ])

    or_estrato = (a * d) / (b * c)

    return or_estrato

or_sem_drc = calcular_or(sem_drc)
or_com_drc = calcular_or(com_drc)

print(f"OR sem DRC: {or_sem_drc:.2f}")
print(f"OR com DRC: {or_com_drc:.2f}")
print(f"OR bruta: {valor_OR_bruta:.2f}")

grupo
caso        21.65
controle    10.63
Name: drc, dtype: float64
drc
0    16.98
1    10.43
Name: exposto, dtype: float64
OR sem DRC: 3.52
OR com DRC: 2.45
OR bruta: 3.10
